# 01 — NightmareNet Quickstart

This notebook gets you from `pip install` to a real distortion + a one-epoch Wake-only training run on a 100-row SST-2 subset, and finishes by plotting the loss curve.

**Runtime:** ~3-5 minutes on CPU, ~60 seconds on a T4 GPU.

**GPU:** Optional. The notebook detects CUDA automatically and falls back to CPU. If you are running on Colab, choose Runtime -> Change runtime type -> T4 GPU for faster execution.

**What you will do:**
1. Install `nightmarenet`
2. Apply a dream distortion and a nightmare distortion to the same input
3. Load 100 rows of SST-2
4. Run a one-epoch Wake-only pipeline (no Dream / Nightmare / Compress)
5. Plot the per-step training loss curve

Skip ahead to `02_benchmark_reproduction.ipynb` once you have this working.

## 1. Install

We install from the local checkout if available, falling back to PyPI.

In [ ]:
import os
import sys
import subprocess

REPO_ROOT_CANDIDATES = ['..', '.', '/content/NightmareNet']
INSTALLED = False
for candidate in REPO_ROOT_CANDIDATES:
    if os.path.exists(os.path.join(candidate, 'pyproject.toml')):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', candidate])
        INSTALLED = True
        break

if not INSTALLED:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nightmarenet'])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'datasets', 'matplotlib'])

## 2. Distortion preview

Apply a dream distortion (mild) and a nightmare distortion (aggressive) to the same input, and print them side-by-side.

In [ ]:
from nightmarenet.distortions.registry import get_registry

registry = get_registry()

text = 'The film was a triumph of restraint and vision, gracefully balanced and quietly powerful.'

dream_out = registry.apply('dream', text, strength=0.3, seed=42)
nightmare_out = registry.apply('nightmare', text, strength=0.8, seed=42)

print('Original:   ', text)
print('Dream  (s=0.3):', dream_out)
print('Nightmare (s=0.8):', nightmare_out)
print()
print('Registered engines:', registry.engine_names)

## 3. Load 100 rows of SST-2

We pull the SST-2 sentiment classification benchmark from the GLUE collection on the Hugging Face Hub and slice the first 100 training rows.

In [ ]:
from datasets import load_dataset

dataset = load_dataset('glue', 'sst2', split='train[:100]')
print(f'Loaded {len(dataset)} rows.')
print('First row:', dataset[0])

## 4. Run a one-epoch Wake-only pipeline

We build a minimal config that runs only the Wake phase (no Dream, no Nightmare, no Compress) so that this notebook completes in a few minutes on a free Colab CPU.

The pipeline emits a per-phase event with the running loss; we collect those into a list and plot them in the next cell.

In [ ]:
import torch
from nightmarenet.pipeline import Pipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

config = {
    'model': {
        'name': 'distilbert-base-uncased',
        'type': 'seq_classification',
        'max_length': 64,
        'num_labels': 2,
    },
    'dataset': {
        'name': 'glue',
        'config': 'sst2',
        'text_column': 'sentence',
        'label_column': 'label',
        'max_samples': 100,
    },
    'training': {
        'wake_epochs': 1,
        'dream_epochs': 0,
        'nightmare_epochs': 0,
        'compress_epochs': 0,
        'num_cycles': 1,
        'batch_size': 8,
        'learning_rate': 5.0e-5,
        'device': device,
    },
    'distortion': {
        'dream_strength': 0.25,
        'nightmare_strength': 0.75,
    },
    'compression': {
        'pruning_ratio': 0.0,
    },
    'evaluation': {
        'strengths': [0.1, 0.5, 0.9],
        'seed': 42,
    },
    'tracking': {'enabled': False},
    'seed': 42,
}

loss_history = []

def collect(event):
    avg_loss = event.get('phase_loss')
    if avg_loss is not None and avg_loss > 0:
        loss_history.append({
            'phase': event.get('current_phase', ''),
            'progress_pct': event.get('progress_pct', 0.0),
            'loss': float(avg_loss),
        })

pipe = Pipeline(config=config, on_event=collect)
pipe.ingest(hf_dataset='glue', hf_subset='sst2')
pipe.prepare()
history = pipe.train()
print(f'Training emitted {len(history)} phase entries.')
for entry in history:
    print(' ', entry)

## 5. Plot the loss curve

Even with one epoch on 100 rows, you should see the loss decrease monotonically. This is the Wake-phase signal that the rest of the cycle (Dream, Nightmare, Compress) builds on.

In [ ]:
import matplotlib.pyplot as plt

if loss_history:
    progress = [entry['progress_pct'] for entry in loss_history]
    losses = [entry['loss'] for entry in loss_history]
    plt.figure(figsize=(8, 4))
    plt.plot(progress, losses, marker='o', color='#06B6D4')
    plt.xlabel('Pipeline progress (%)')
    plt.ylabel('Phase loss')
    plt.title('Wake-only training loss (SST-2, 100 rows, 1 epoch)')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    fallback = [entry.get('avg_loss', 0.0) for entry in history]
    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(fallback) + 1), fallback, marker='o', color='#06B6D4')
    plt.xlabel('Phase index')
    plt.ylabel('Avg loss')
    plt.title('Per-phase loss (Wake only)')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## Next steps

You have just run the simplest possible NightmareNet workflow. To go deeper:

- **`02_benchmark_reproduction.ipynb`** — reproduce the SST-2 benchmark from the README. Runs both `configs/benchmark_sst2_baseline.yaml` (Wake-only) and `configs/benchmark_sst2.yaml` (full 4-phase cycle) and compares them.
- **`03_custom_distortions.ipynb`** — write your own distortion plugin, register it via `@register_distortion`, and evaluate a model with it.

For the full architecture, see [`README.md`](../README.md) and [`docs/architecture/PRD.md`](../docs/architecture/PRD.md).